In [ ]:
import os, json, urllib.request, pathlib

API_KEY = ""

req = pathlib.Path("/content/request.json").read_bytes()
print("request size:", round(len(req)/1024, 1), "KB")

r = urllib.request.Request(
    "https://health.api.nvidia.com/v1/biology/ipd/rfd4-proteina/generative",
    data=req,
    headers={"Authorization": f"Bearer {API_KEY}",
             "Content-Type": "application/json",
             "NVCF-POLL-SECONDS": "300"},
    method="POST")

try:
    with urllib.request.urlopen(r, timeout=600) as resp:
        result = json.load(resp)
except urllib.error.HTTPError as e:
    print("HTTP", e.code)
    print(e.read().decode()[:2000])
    raise

print(f"\n{result['task']} in {result['duration_ms']/1000:.1f}s\n")
seqs = []
for d in result["designs"]:
    s = d["sequence"]
    seqs.append(s)
    pathlib.Path(d["structure"]["filename"]).write_text(d["structure"]["content"])
    print(d["design_id"], len(s), s)

pathlib.Path("/content/tfr1_designs.txt").write_text("\n".join(seqs))
print("\n", len(seqs), "designs saved to /content/tfr1_designs.txt")

request size: 203.7 KB

generation in 49.5s

design_0 272 AVASIKSQVLAGLKAQLSPAEFAVAKAVVEAVVAANPDAPPEVLLGDIQAALEAAFPSKSFSISQAADGTISVVVSDKIQVKDSAQNSVIIVDKNGRLVYLVENPGGYVAYSKAATVTGKLVHANFGTKKDFEDLYTPVNGSIVIVRAGKITFAEKVANAESLNAIGVLIYMDQTKFPIVNAELSFFGHAHLGTGDPYTPGFPSFNHTQFPPSRSSGLPNIPVQTISRAAAEKLFGNMEGDCPSDWKTDSTCRMVTSESKNVKLTVSNVLKE
design_1 257 AVVAALAAALAAVKAGDVAAALAALDAAVAAGADAAAVAAIKAAIAAGDLAAAAAALEAALAKIQVKDSAQNSVIIVDKNGRLVYLVENPGGYVAYSKAATVTGKLVHANFGTKKDFEDLYTPVNGSIVIVRAGKITFAEKVANAESLNAIGVLIYMDQTKFPIVNAELSFFGHAHLGTGDPYTPGFPSFNHTQFPPSRSSGLPNIPVQTISRAAAEKLFGNMEGDCPSDWKTDSTCRMVTSESKNVKLTVSNVLKE
design_2 278 TSLSISDCPNLTSLSISDCPNLTSLSISDCPNLTSLSISDCPNLTSLSISDCPNLTSLSISDCPNLTSLSISDCPKLTSLSISKIQVKDSAQNSVIIVDKNGRLVYLVENPGGYVAYSKAATVTGKLVHANFGTKKDFEDLYTPVNGSIVIVRAGKITFAEKVANAESLNAIGVLIYMDQTKFPIVNAELSFFGHAHLGTGDPYTPGFPSFNHTQFPPSRSSGLPNIPVQTISRAAAEKLFGNMEGDCPSDWKTDSTCRMVTSESKNVKLTVSNVLKE
design_3 259 SLLSPQTYKVEYILPDGSVIEESISASNVFEAQVKAEEKGKELGAVKVKIYDSDGQLVSDIDLSKIQVKDSAQNSVIIVDKNGRLVYLVENPG

In [ ]:
!pip install -q py3Dmol
import py3Dmol, glob, pathlib

files = sorted(glob.glob("/content/*.cif")) + sorted(glob.glob("/content/*.pdb"))
files = [f for f in files if "request" not in f and "annotated" not in f]
print("\n".join(files))

f = files[0]                      # design_0
data = pathlib.Path(f).read_text()
fmt = "cif" if f.endswith(".cif") else "pdb"

v = py3Dmol.view(width=900, height=650)
v.addModel(data, fmt)
v.setStyle({}, {"cartoon": {"color": "lightgrey"}})          # target
v.setStyle({"chain": "A"}, {"cartoon": {"color": "spectrum"}})  # binder
for r in (206, 209, 374):                                     # hotspots
    v.addStyle({"chain": "B", "resi": r},
               {"stick": {"colorscheme": "orangeCarbon", "radius": 0.25}})
v.zoomTo()
v.show()

/content/design_0.cif
/content/design_1.cif
/content/design_2.cif
/content/design_3.cif
/content/design_4.cif
/content/design_5.cif
/content/design_6.cif
/content/design_7.cif
/content/design_8.cif
/content/design_9.cif


3Dmol.js failed to load for some reason. Please check your browser console for error messages.